<a href="https://colab.research.google.com/github/sarmad-usman/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarmad-usman/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
import pandas as pd

# Placeholder: Load your raw data here
# For example:
# raw_data = pd.read_csv('your_data.csv')
# Or if already loaded:
# raw_data = ... # Assume raw_data DataFrame is available from previous steps

# --- Start Feature Engineering ---

# Example: Create a dummy DataFrame if raw_data is not available
try:
    raw_data
except NameError:
    print("Warning: 'raw_data' DataFrame not found. Creating a dummy for demonstration.")
    raw_data = pd.DataFrame({
        'id': range(10),
        'feature_a': [10, 20, None, 40, 50, 60, 70, None, 90, 100],
        'feature_b': ['cat_X', 'cat_Y', 'cat_X', 'cat_Z', 'cat_Y', 'cat_X', 'cat_Z', 'cat_Y', 'cat_X', 'cat_Z'],
        'feature_c': [1.1, 2.2, 3.3, 4.4, 5.5, 6.6, 7.7, 8.8, 9.9, 10.0]
    })

feature_vector = raw_data.copy()

# Engineered Features:
# Add your custom engineered features here.
# Example: feature_vector['new_feature'] = feature_vector['feature_a'] * 2

# Categorical Handling:
# Convert categorical features into a numerical representation (e.g., one-hot encoding).
# Example:
# categorical_cols = ['feature_b']
# feature_vector = pd.get_dummies(feature_vector, columns=categorical_cols, drop_first=True)

# Missing Value Fills:
# Handle any missing values (e.g., imputation with mean, median, or a constant).
# Example:
# numerical_cols_to_fill = ['feature_a']
# for col in numerical_cols_to_fill:
#     feature_vector[col].fillna(feature_vector[col].median(), inplace=True)

# Remove original features if they have been transformed or replaced
# Example: feature_vector = feature_vector.drop(columns=['original_feature_that_was_encoded'])

# --- End Feature Engineering ---

print("Feature Vector Head:")
print(feature_vector.head())
print("\nFeature Vector Info:")
feature_vector.info()

Feature Vector Head:
   id  feature_a feature_b  feature_c
0   0       10.0     cat_X        1.1
1   1       20.0     cat_Y        2.2
2   2        NaN     cat_X        3.3
3   3       40.0     cat_Z        4.4
4   4       50.0     cat_Y        5.5

Feature Vector Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         10 non-null     int64  
 1   feature_a  8 non-null      float64
 2   feature_b  10 non-null     object 
 3   feature_c  10 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 452.0+ bytes


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [2]:
print("""
| Feature     | Meaning                        | Missing Value Handling     | Categorical Handling            | Available Before Prediction |
|-------------|--------------------------------|----------------------------|---------------------------------|-----------------------------|
| `id`        | Unique identifier for each entry | N/A (no missing values)    | N/A (numerical, not categorical) | Yes                         |
| `feature_a` | Generic numerical feature      | Filled with median (example) | N/A (numerical)                 | Yes                         |
| `feature_b` | Generic categorical feature    | N/A (no missing values in dummy data) | One-hot encoded (example)         | Yes                         |
| `feature_c` | Another generic numerical feature | N/A (no missing values)    | N/A (numerical)                 | Yes                         |
""")


| Feature     | Meaning                        | Missing Value Handling     | Categorical Handling            | Available Before Prediction |
|-------------|--------------------------------|----------------------------|---------------------------------|-----------------------------|
| `id`        | Unique identifier for each entry | N/A (no missing values)    | N/A (numerical, not categorical) | Yes                         |
| `feature_a` | Generic numerical feature      | Filled with median (example) | N/A (numerical)                 | Yes                         |
| `feature_b` | Generic categorical feature    | N/A (no missing values in dummy data) | One-hot encoded (example)         | Yes                         |
| `feature_c` | Another generic numerical feature | N/A (no missing values)    | N/A (numerical)                 | Yes                         |



## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [3]:
# Assuming 'feature_vector' from the previous section is available.
# If not, you might need to re-run the previous cell or load data here.

print("### 3. The leakage hunt ###\n")

# --- Step 1: Define a hypothetical target variable (for demonstration) ---
# In a real scenario, this would be your actual label/target variable.
# For this example, let's create a synthetic target that might accidentally
# show some correlation with 'id' or other features if not careful.

# Let's create a dummy target that's somewhat related to 'feature_a' and 'id'
# to simulate a potential leakage scenario.
if 'feature_vector' in locals() and not feature_vector.empty:
    # Ensure the target has the same index as the feature_vector
    dummy_target = pd.Series(feature_vector['feature_a'].fillna(feature_vector['feature_a'].median()) +
                             (feature_vector['id'] * 0.5) +
                             np.random.randn(len(feature_vector)) * 2,
                             index=feature_vector.index,
                             name='dummy_target')
    print("Created a synthetic 'dummy_target' for leakage demonstration.\n")

    # --- Step 2: Combine features and target for correlation analysis ---
    # Drop 'id' from correlation if it's just an identifier and not a feature itself.
    # However, for leakage, we might check its correlation too.
    df_for_leakage_check = feature_vector.copy()

    # If 'id' is purely an identifier and not meant as a feature, it's often dropped
    # for modeling, but its correlation with target could still indicate data ordering issues.
    if 'id' in df_for_leakage_check.columns:
        df_for_leakage_check = df_for_leakage_check.drop(columns=['id'])

    # Ensure all columns are numeric for correlation calculation
    numeric_df = df_for_leakage_check.select_dtypes(include=np.number)

    # Add the target to the numeric DataFrame
    if 'dummy_target' in locals():
        numeric_df['dummy_target'] = dummy_target

        # --- Step 3: Calculate correlations ---
        print("Correlations with 'dummy_target':\n")
        correlations = numeric_df.corr()['dummy_target'].drop('dummy_target')
        print(correlations.sort_values(ascending=False))

        print("\n--- Interpretation ---")
        print("High correlations (e.g., > 0.8 or < -0.8) between a feature and the target")
        print("can sometimes indicate data leakage, especially if the feature wouldn't ")
        print("logically be available at the time of prediction or if it's derived from the target itself.")
        print("Review any highly correlated features to ensure they don't introduce leakage.")
    else:
        print("Could not create 'dummy_target' as 'feature_vector' was not found or was empty.")
        print("Please ensure the previous cell runs successfully to create 'feature_vector'.")
else:
    print("Feature vector not found. Please ensure the 'Build the feature vector' cell runs successfully.")

import numpy as np

### 3. The leakage hunt ###



NameError: name 'np' is not defined

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
print("""
| Excluded Feature | Reason for Exclusion                                     |
|------------------|----------------------------------------------------------|
| `original_timestamp` | Would lead to data leakage if used for future prediction. |
| `customer_id`    | Purely an identifier; no predictive power, potential privacy concern. |
| `target_label_in_raw` | Direct leakage of the target variable.                   |
| `future_event_flag` | Contains information not available at prediction time.   |
| `raw_text_description` | Too high cardinality for current model, requires NLP.  |
| `unreliable_sensor_data` | Low data quality, many missing values, inconsistent readings. |
""")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.